# FoundationPose Evaluation — Colab GPU

**Ejecutar DESPUES de `00_colab_setup.ipynb`**

Este notebook:
1. Instala FoundationPose desde el repo oficial de NVIDIA
2. Descarga pesos pre-entrenados
3. Ejecuta inferencia en YCB-V y T-LESS
4. Calcula metricas BOP (VSD, MSSD, MSPD)
5. Guarda resultados en Google Drive

**Requiere:** GPU T4 o superior (Runtime > Change runtime type > T4 GPU)

In [ ]:
import torch, os
assert torch.cuda.is_available(), "GPU requerida! Ve a Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

REPO_DIR = "/content/repo_tfm"
assert os.path.exists(REPO_DIR), "Ejecuta 00_colab_setup.ipynb primero"
os.chdir(REPO_DIR)

import sys; sys.path.insert(0, REPO_DIR)

## 1. Instalar FoundationPose

In [ ]:
FP_DIR = "/content/FoundationPose"

if not os.path.exists(FP_DIR):
    !git clone https://github.com/NVlabs/FoundationPose.git {FP_DIR}
    print("FoundationPose clonado")
else:
    print("FoundationPose ya existe")

# Instalar dependencias de FoundationPose
!cd {FP_DIR} && pip install -q -e .

# Dependencias adicionales que pueda necesitar
!pip install -q nvdiffrast trimesh pyrender
!pip install -q kaolin -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.1.0_cu121.html 2>/dev/null || echo "kaolin install may need manual setup"

print("\nFoundationPose instalado")

In [ ]:
# Descargar pesos pre-entrenados
WEIGHTS_DIR = "/content/drive/MyDrive/TFM/weights/foundationpose"
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Los pesos oficiales de FoundationPose
# Ref: https://github.com/NVlabs/FoundationPose#pre-trained-weights
FP_CKPTS = {
    "refine_model": "https://drive.google.com/uc?id=1hH_mTXHcG51HLCAFJjjCp4JFxoPjGqvd",
    "score_model": "https://drive.google.com/uc?id=1S0Gz9U6Y6XnXw1SPQNLHFKMzGe2WAXRZ",
}

print("Pesos de FoundationPose:")
print("Nota: Los pesos oficiales requieren descarga manual desde NVIDIA.")
print(f"Guardalos en: {WEIGHTS_DIR}")
print("\nAlternativamente, el wrapper puede usar pesos de HuggingFace.")
print("Consulta: https://github.com/NVlabs/FoundationPose#pre-trained-weights")

## 2. Configurar evaluacion BOP

In [ ]:
# BOP Toolkit
!pip install -q bop_toolkit_lib

import numpy as np
import json
import time
from pathlib import Path

# Verificar datasets
DATA_DIR = f"{REPO_DIR}/data/datasets"
for ds in ['ycbv', 'tless']:
    p = Path(DATA_DIR) / ds
    if p.exists():
        models = list((p / 'models').glob('obj_*.ply')) if (p / 'models').exists() else []
        print(f"{ds}: {len(models)} modelos 3D")
        # Verificar imagenes de test
        test_dirs = [d for d in p.iterdir() if d.is_dir() and 'test' in d.name]
        for td in test_dirs:
            scenes = [d for d in td.iterdir() if d.is_dir()]
            print(f"  {td.name}: {len(scenes)} escenas")
    else:
        print(f"{ds}: NO encontrado")

## 3. Ejecutar FoundationPose en YCB-V

In [ ]:
from src.perception.foundation_pose import FoundationPoseEstimator
from src.utils.dataset_loader import BOPDataset
from src.utils.metrics import compute_add, compute_adds
from src.utils.visualization import visualize_pose_comparison
from tqdm.notebook import tqdm

# Cargar dataset
ycbv = BOPDataset(f"{DATA_DIR}/ycbv", split="test")
scenes = ycbv.get_scene_ids()
print(f"YCB-V: {len(scenes)} escenas")

# Inicializar estimador
estimator = FoundationPoseEstimator(
    weights_dir=WEIGHTS_DIR,
    device="cuda",
)

# Cargar modelos 3D
models_dir = f"{DATA_DIR}/ycbv/models"
model_paths = sorted(Path(models_dir).glob("obj_*.ply"))
print(f"Modelos 3D: {len(model_paths)}")

In [ ]:
# Ejecutar inferencia
results_ycbv = []
timing_total = 0
n_evaluated = 0

MAX_SCENES = 5  # Limitar para primera prueba (cambiar a None para todas)
eval_scenes = scenes[:MAX_SCENES] if MAX_SCENES else scenes

for scene_id in tqdm(eval_scenes, desc="Scenes"):
    n_images = ycbv.get_num_images(scene_id)
    cameras = ycbv.load_scene_camera(scene_id)
    gt_poses = ycbv.load_scene_gt(scene_id)
    
    for img_id in range(min(n_images, 50)):  # Max 50 imgs/scene
        try:
            sample = ycbv.load_sample(scene_id, img_id)
            
            t0 = time.time()
            pred = estimator.estimate_pose(
                rgb=sample['rgb'],
                depth=sample['depth'],
                mask=sample.get('mask'),
                K=sample['cam_K'],
            )
            elapsed = time.time() - t0
            timing_total += elapsed
            n_evaluated += 1
            
            results_ycbv.append({
                'scene_id': scene_id,
                'img_id': img_id,
                'R_pred': pred['R'].tolist(),
                't_pred': pred['t'].tolist(),
                'score': pred['score'],
                'time_s': elapsed,
            })
        except Exception as e:
            print(f"  Error scene {scene_id} img {img_id}: {e}")

avg_time = timing_total / max(n_evaluated, 1)
print(f"\nEvaluadas {n_evaluated} imagenes")
print(f"Tiempo promedio: {avg_time*1000:.1f} ms/imagen")
print(f"FPS: {1/avg_time:.1f}")

## 4. Calcular Metricas BOP

In [ ]:
from src.perception.evaluator import PoseEvaluator

evaluator = PoseEvaluator(
    dataset_root=f"{DATA_DIR}/ycbv",
    models_dir=str(models_dir),
)

# Calcular metricas
metrics = evaluator.evaluate(
    predictions=results_ycbv,
    metrics=['add', 'adds', 'vsd', 'mssd', 'mspd'],
)

print("=" * 60)
print("FoundationPose — YCB-Video Results")
print("=" * 60)
for metric_name, value in metrics.items():
    print(f"  {metric_name}: {value:.4f}")

In [ ]:
# Visualizar ejemplos
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, ax_row in enumerate(axes):
    if i * 4 >= len(results_ycbv): break
    for j, ax in enumerate(ax_row):
        idx = i * 4 + j
        if idx >= len(results_ycbv): break
        r = results_ycbv[idx]
        sample = ycbv.load_sample(r['scene_id'], r['img_id'])
        ax.imshow(sample['rgb'])
        ax.set_title(f"S{r['scene_id']}/I{r['img_id']} score={r['score']:.2f}")
        ax.axis('off')

plt.suptitle('FoundationPose — YCB-V Predictions', fontsize=16)
plt.tight_layout()
plt.show()

## 5. Ejecutar en T-LESS (si esta disponible)

In [ ]:
# Repetir para T-LESS (mismo flujo)
tless_test_path = Path(f"{DATA_DIR}/tless/test_primesense")

if tless_test_path.exists():
    tless = BOPDataset(f"{DATA_DIR}/tless", split="test_primesense")
    tless_scenes = tless.get_scene_ids()
    print(f"T-LESS: {len(tless_scenes)} escenas")
    
    # Evaluar (mismo patron que YCB-V)
    results_tless = []
    eval_tless = tless_scenes[:3]  # Primeras 3 para prueba
    
    for scene_id in tqdm(eval_tless, desc="T-LESS"):
        n_images = tless.get_num_images(scene_id)
        for img_id in range(min(n_images, 50)):
            try:
                sample = tless.load_sample(scene_id, img_id)
                pred = estimator.estimate_pose(
                    rgb=sample['rgb'],
                    depth=sample['depth'],
                    mask=sample.get('mask'),
                    K=sample['cam_K'],
                )
                results_tless.append({
                    'scene_id': scene_id,
                    'img_id': img_id,
                    'R_pred': pred['R'].tolist(),
                    't_pred': pred['t'].tolist(),
                    'score': pred['score'],
                })
            except Exception as e:
                pass
    
    print(f"T-LESS: {len(results_tless)} predicciones")
else:
    print("T-LESS test images no disponibles.")
    print("Ejecuta 00_colab_setup.ipynb para descargarlas.")

## 6. Guardar resultados

In [ ]:
import json
from datetime import datetime

USE_GDRIVE = os.path.exists('/content/drive/MyDrive')
base_dir = "/content/drive/MyDrive/TFM/experiments" if USE_GDRIVE else f"{REPO_DIR}/experiments"

# YCB-V results
exp_dir = f"{base_dir}/foundationpose_ycbv"
os.makedirs(exp_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
result_file = f"{exp_dir}/results_{timestamp}.json"

output = {
    'method': 'FoundationPose',
    'dataset': 'ycbv',
    'n_images': n_evaluated,
    'avg_time_ms': avg_time * 1000,
    'fps': 1 / avg_time if avg_time > 0 else 0,
    'gpu': torch.cuda.get_device_name(0),
    'predictions': results_ycbv,
}

with open(result_file, 'w') as f:
    json.dump(output, f, indent=2)

print(f"Resultados guardados: {result_file}")

# T-LESS results (si existen)
if 'results_tless' in dir() and results_tless:
    exp_dir_tless = f"{base_dir}/foundationpose_tless"
    os.makedirs(exp_dir_tless, exist_ok=True)
    with open(f"{exp_dir_tless}/results_{timestamp}.json", 'w') as f:
        json.dump({'method': 'FoundationPose', 'dataset': 'tless', 'predictions': results_tless}, f, indent=2)
    print(f"T-LESS guardado: {exp_dir_tless}/results_{timestamp}.json")

print("\nListo! Resultados persistidos en Google Drive.")
print("Continua con 02_gdrnet_eval.ipynb para el baseline comparativo.")